# Transformer Pipeline for the Coin and Flower Processes

The transformer counterpart of `updated_pipeline_asymmetric_process.ipynb` (the discrete-memory GRU). Nothing here re-implements the pipeline: the samplers, closed forms and true ε-machines come from `processes.py`, the model from `models.py`, training from `training.py` and the extractors from `extraction.py`. The notebook holds only the configuration, the process spec built from it, and the plotting.

The model is the **`DiscreteCausalDecoder`** (`models.py`): prediction is forced through a K-way straight-through bottleneck, so a position's causal state is `argmax(state_logits)`. Occupancy and `S_emp`, the emission table `P(token | state)`, the free-running state-to-state transition matrix and an empirical symbolic machine `(next_state, emission_probs)` are all read off the model directly.

Two arms, both tested on both processes:

- **forward** — tril mask, predicts the next token; compared with C+, `occ_fw`, `T_theory_fw` and `true_machine(..., "forward")`.
- **backward** — triu mask, predicts the previous token; compared with C−, `occ_bw`, `T_theory_bw` (`processes.coin_rev_transition_matrix` / `flower_rev_transition_matrix`) and `true_machine(..., "backward")`.

The learned transition matrix is compared with the closed form of its arm by `extraction.compare_transition_matrix`: learned states are mapped onto the theoretical ones by their emission rows (the identified object), so a causal state the bottleneck split into several is re-merged before the cell-by-cell comparison, and a theoretical state no learned state reaches is reported as missing. `transition_matrix_extraction` already records a backward model's matrix one step back in time, which is the convention of the backward closed forms, so both arms compare without transposition.

`run_pipeline(cfg, process, mode)` runs one arm; `plot_arm(results)` draws everything for it; `plot_transition_comparison_arms` and `plot_complexity` put the two arms side by side. Cross-entropies are in **bits/token** throughout, as in the modules.

In [ ]:

import os
import math
import time
import random
import copy
from dataclasses import dataclass, asdict, replace
from typing import Dict, List, Tuple, Optional

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.utils.data as tud

# Everything that defines the experiment lives in the modules; this notebook only configures, runs and plots.
from processes import (SequenceDataset, generate, make_dice, true_machine, entropy_bits,
                       coin_complexity, flower_complexity,
                       entropy_rate_coin, flower_entropy_rate,
                       causal_state_count, causal_state_occupancy,
                       coin_transition_matrix, coin_rev_transition_matrix,
                       flower_transition_matrix, flower_rev_transition_matrix,
                       coin_tag, flower_tag)
from models import DiscreteCausalDecoder                         # built by training.train_model
from training import (train_model, split_loader, eval_ce, set_seed, to_cpu_for_analysis,
                      diagnose_divergence, cleanup, quiet)
from extraction import (causal_state_report, transition_matrix_extraction,
                        match_permutation, compare_transition_matrix)
from schedules import parse_tau

print(f"Torch version: {torch.__version__}")
print(f"Accelerators: cuda={torch.cuda.is_available()}  mps={torch.backends.mps.is_available()}")

In [ ]:

@dataclass
class ExperimentConfig:
    # --- Process parameters: run_pipeline(cfg, process, mode) with process in {"coin", "flower"} ---
    # coin: hidden coin 0 -> 1 w.p. p, 1 -> 0 w.p. q; tokens {0, 1, 2}
    asym_p: float = 0.70
    asym_q: float = 0.80
    coin_burn_in: int = 200

    # flower: n dice with m faces; tokens 0..n-1 select a die, n..n+m-1 are its roll
    flower_n: int = 3
    flower_m: int = 8
    flower_dice_seed: int = 0      # the dice DEFINE the flower process, so they get their own seed
    flower_burn_in: int = 4        # iid across cycles, so the burn-in is belt-and-braces

    data_seed: int = 1             # rng handed to processes.generate

    # --- Data: num_samples sequences of seq_len tokens (processes.SequenceDataset), every token trained on ---
    seq_len: int = 300             # also the transformer's context (max_len) and the free-running window
    num_samples: int = 500
    batch_size: int = 32
    test_ratio: float = 0.20       # seeded split, training.split_loader

    # --- Model (models.py) ---
    # V (vocabulary), K (discrete-state budget) and S (state_dim) follow the process:
    # V = 3 for coin, n + m for flower, K = num_states_mult * V, S = state_dim_mult * V.
    # All three are resolved per run by resolve_cfg below; the values here are the coin ones.
    d_model: int = 32
    n_layers: int = 2
    num_states_mult: int = 4       # K is a state BUDGET, not an estimate: the bottleneck needs slack
    state_dim_mult: int = 1        # S carries no expressive power (state_matrix @ emission is one (K,V) map)
    num_states: int = 12
    state_dim: int = 3
    symbol_vocab_size: int = 3
    tau: float | str = "geom:0.5:5"      # straight-through temperature: a float, "const:X" or "geom:A:B" (schedules.py)
    usage_beta: Optional[float] = None   # None -> 1 / (batch_size * seq_len); models.DiscreteCausalDecoder.usage_penalty

    # --- Training (training.py -> lightning) ---
    max_epochs: int = 150
    learning_rate: float = 1e-3
    weight_decay: float = 0.01     # what keeps the deterministic transitions from diverging (models._Decoder.configure_optimizers)
    accelerator: str = "auto"      # "auto" picks MPS on a Mac (fast, not bit-reproducible); "cpu" is exactly repeatable
    val_every_n_steps: int = 25
    quiet_training: bool = True    # swallow Lightning's per-model chatter (re-emitted if training raises)

    # --- Extraction (extraction.py) ---
    ana_batch: int = 32
    state_min_pos: int = 5         # drop the short-context end of each sequence when reading states
    conv_tol: float = 0.10         # |CE - H_inf| (bits) above this = not converged
    trans_total_run: int = 5000    # free-running generation steps for the state-transition matrix

    random_seed: int = 0           # split + initialisation

def resolve_cfg(cfg: ExperimentConfig, num_token: int) -> ExperimentConfig:
    """One cfg per run: V is the process's token count, K = num_states_mult * V and
    S = state_dim_mult * V.  usage_beta = 1/(batch * seq_len) unless set explicitly --
    it must stay under the optimisation cliff (~6e-4 at lr 1e-3) above which the
    bottleneck collapses to one state, and the cliff moves with batch and seq_len."""
    V = int(num_token)
    beta = (1.0 / (cfg.batch_size * cfg.seq_len) if cfg.usage_beta is None
            else float(cfg.usage_beta))
    return replace(cfg, symbol_vocab_size=V, num_states=cfg.num_states_mult * V,
                   state_dim=cfg.state_dim_mult * V, usage_beta=beta)

cfg = ExperimentConfig()
set_seed(cfg.random_seed)
parse_tau(cfg.tau)                 # validate the schedule now, not a minute into the first fit

print("Experiment configuration:")
print(pd.Series(asdict(cfg)))

In [ ]:

ARM = {"forward": "fw", "backward": "bw"}

def process_spec(cfg: ExperimentConfig, process: str) -> Dict:
    """Everything the pipeline needs to know about the chosen process, for BOTH arms,
    from the notebook cfg alone.

    The sampler parameters and every closed form (entropy rate, C+/C-, causal-state
    counts, stationary occupancies, state-transition matrices and the true
    epsilon-machines with emissions) come from processes.py, so the data and the
    theory it is compared against are computed from the same numbers.

    Vocabulary:  coin  V = 3,   flower  V = n + m.
    """
    if process == "coin":
        p, q = cfg.asym_p, cfg.asym_q
        params = {"p": p, "q": q}
        C_plus, C_minus = coin_complexity(p, q)
        spec = {
            "kind": "coin", "tag": coin_tag(p, q), "params": params,
            "num_token": 3, "burn_in": cfg.coin_burn_in,
            "entropy_rate": entropy_rate_coin(p, q),          # bits/token, both arms
            "C_plus": C_plus, "C_minus": C_minus,
            "true_k_fw": causal_state_count("coin", "forward"),
            "true_k_bw": causal_state_count("coin", "backward"),
            "occ_fw": causal_state_occupancy("coin", "forward",  p=p, q=q),
            "occ_bw": causal_state_occupancy("coin", "backward", p=p, q=q),
            "T_theory_fw": coin_transition_matrix(p, q),
            "T_theory_bw": coin_rev_transition_matrix(p, q),
        }
    elif process == "flower":
        n, m = cfg.flower_n, cfg.flower_m
        dice = make_dice(n, m, cfg.flower_dice_seed)
        params = {"n": n, "m": m, "dice_probs": dice}
        C_plus, C_minus = flower_complexity(n, m, dice)
        spec = {
            "kind": "flower", "tag": flower_tag(n, m), "params": params,
            "num_token": n + m, "burn_in": cfg.flower_burn_in,
            "entropy_rate": flower_entropy_rate(n, m, dice),  # bits/token, both arms
            "C_plus": C_plus, "C_minus": C_minus,
            "true_k_fw": causal_state_count("flower", "forward", n=n, m=m),
            "true_k_bw": causal_state_count("flower", "backward", n=n, m=m, dice_probs=dice),
            "occ_fw": causal_state_occupancy("flower", "forward", n=n),
            "occ_bw": causal_state_occupancy("flower", "backward", n=n, m=m, dice_probs=dice),
            "T_theory_fw": flower_transition_matrix(n),
            "T_theory_bw": flower_rev_transition_matrix(n, m, dice),
        }
    else:
        raise ValueError(f"unknown process {process!r}; expected 'coin' or 'flower'")

    # The true epsilon-machines in (next_state, emission_probs) form, one per arm.
    spec["true_machine_fw"] = true_machine(spec["kind"], params, "forward")
    spec["true_machine_bw"] = true_machine(spec["kind"], params, "backward")
    return spec

def theory_for_arm(spec: Dict, mode: str) -> Dict:
    """The closed forms the model in this arm is compared against."""
    arm = ARM[mode]
    return {
        "arm": arm, "names": state_names(spec, mode),
        "C": spec["C_plus"] if arm == "fw" else spec["C_minus"],
        "true_k": spec[f"true_k_{arm}"],
        "occupancy": spec[f"occ_{arm}"],
        "T_theory": spec[f"T_theory_{arm}"],
        "true_machine": spec[f"true_machine_{arm}"],
    }

def state_names(spec: Dict, mode: str) -> List[str]:
    """Names of the theoretical states in the order of causal_state_occupancy / true_machine."""
    if spec["kind"] == "coin":
        return ["tails", "heads"] if mode == "forward" else ["tok 0", "tok 1", "tok 2"]
    n = spec["params"]["n"]
    if mode == "forward":
        return ["R"] + [f"die {i}" for i in range(n)]
    return ["S"] + [f"class {c}" for c in range(spec["true_k_bw"] - 1)]

def machine_state_transition(machine: Dict[str, np.ndarray]) -> np.ndarray:
    """(K, K) state-to-state matrix induced by an emission-labelled machine."""
    next_state = np.asarray(machine["next_state"], dtype=np.int64)
    emission_probs = np.asarray(machine["emission_probs"], dtype=np.float64)
    num_states = next_state.shape[0]
    T = np.zeros((num_states, num_states), dtype=np.float64)
    for s in range(num_states):
        for x in range(emission_probs.shape[1]):
            T[s, int(next_state[s, x])] += float(emission_probs[s, x])
    return T

def stationary_distribution_from_machine(machine: Dict[str, np.ndarray]) -> np.ndarray:
    T = machine_state_transition(machine)
    num_states = T.shape[0]
    A = T.T - np.eye(num_states)
    A[-1, :] = 1.0
    b = np.zeros(num_states, dtype=np.float64)
    b[-1] = 1.0
    pi = np.linalg.solve(A, b)
    pi = np.clip(pi, 0.0, None)
    pi /= max(pi.sum(), 1e-12)
    return pi

def true_state_from_past(true_machine_: Dict[str, np.ndarray], past_seq: np.ndarray) -> int:
    """Causal state after a past window.  Every true machine is synchronised by the last
    symbol, so it is next_state[., last symbol], the same for every row."""
    return int(true_machine_["next_state"][0, int(past_seq[-1])])

In [ ]:

def arm_inputs(seqs: np.ndarray, mode: str) -> np.ndarray:
    """The tokens a model of this arm sees, in its own alignment (models._Decoder._split):
    x[:-1] forward, x[1:] backward."""
    seqs = np.asarray(seqs, dtype=np.int64)
    return seqs[:, :-1] if mode == "forward" else seqs[:, 1:]

@torch.no_grad()
def hard_state_trajectories(model, loader):
    """(states, tokens), (N, T) each: the bottleneck state and the model's input token at
    every position, in the arm's own alignment.  Discrete model only."""
    model.eval()
    device = next(model.parameters()).device
    states, tokens = [], []
    for batch in loader:
        inputs, _ = model._split(batch, model.mode)
        model(inputs.to(device))
        states.append(model.last_states.detach().cpu().numpy())
        tokens.append(inputs.detach().cpu().numpy())
    return np.concatenate(states, axis=0), np.concatenate(tokens, axis=0)

def kept_positions(mode: str, T: int, min_pos: int) -> slice:
    """extraction.causal_state_report's rule: drop the short-context end of each sequence,
    which is the START of a forward model and the FINISH of a backward one."""
    return slice(min_pos, None) if mode == "forward" else slice(0, max(T - min_pos, 1))

def empirical_symbolic_machine(model, loader, min_pos: int) -> Dict:
    """The unifilar machine the discrete model has BECOME, read off its state trajectories.

    A transformer's state is a function of the whole context, so this is an estimate:
    next_state[s, x] is the majority successor of (state s, token x) and determinism[s, x]
    the fraction of visits that agree with it -- 1 everywhere for a genuinely unifilar
    machine.  Forward the step is (s_t, x_{t+1}) -> s_{t+1}; backward it is one step back
    in time, (s_t, x_{t-1}) -> s_{t-1}, matching processes.true_machine(..., "backward").
    emission_probs is model.emission_table() over the visited states, which are relabelled
    0..k-1 (state_ids maps back to the bottleneck's indices, relabel the other way).
    """
    states, tokens = hard_state_trajectories(model, loader)
    K, V = int(model.n_states), int(model.token_size)
    keep = kept_positions(model.mode, states.shape[1], min_pos)
    states, tokens = states[:, keep], tokens[:, keep]

    if model.mode == "forward":
        s_from, x, s_to = states[:, :-1], tokens[:, 1:], states[:, 1:]
    else:
        s_from, x, s_to = states[:, 1:], tokens[:, :-1], states[:, :-1]
    triples = np.zeros((K, V, K), dtype=np.float64)
    np.add.at(triples, (s_from.ravel(), x.ravel(), s_to.ravel()), 1.0)

    counts = np.bincount(states.ravel(), minlength=K).astype(np.float64)
    visited = np.flatnonzero(counts > 0)
    relabel = np.full(K, -1, dtype=np.int64)
    relabel[visited] = np.arange(len(visited))

    k = len(visited)
    next_state = np.zeros((k, V), dtype=np.int64)
    determinism = np.full((k, V), np.nan)
    for i, s in enumerate(visited):
        for symbol in range(V):
            row = triples[s, symbol]
            if row.sum() > 0:
                j = int(row.argmax())
                next_state[i, symbol] = relabel[j]
                determinism[i, symbol] = row[j] / row.sum()
            else:
                next_state[i, symbol] = i          # (s, x) never observed: a self-loop under a ~0 emission
    emission_probs = model.emission_table().detach().cpu().numpy()[visited]

    return {
        "next_state": next_state, "emission_probs": emission_probs,
        "determinism": determinism, "state_ids": visited, "relabel": relabel,
        "counts": counts[visited], "init_probs": counts[visited] / counts[visited].sum(),
        "pair_counts": triples[visited].sum(axis=-1),
    }

def symbolic_cross_entropy(machine: Dict, tokens: np.ndarray, start_states, mode: str, t0: int) -> float:
    """Bits/token of a unifilar machine run along the arm over the model's input tokens,
    started from start_states at position t0: forward over positions t0+1.., backward
    over t0-1..0.  The same scorer serves the empirical machine and the true one."""
    next_state = np.asarray(machine["next_state"], dtype=np.int64)
    emission_probs = np.asarray(machine["emission_probs"], dtype=np.float64)
    total_bits, total_steps = 0.0, 0
    for row, s0 in zip(tokens, start_states):
        s = int(s0)
        path = row[t0 + 1:] if mode == "forward" else row[:t0][::-1]
        for x in path:
            total_bits -= math.log2(max(float(emission_probs[s, int(x)]), 1e-12))
            s = int(next_state[s, int(x)])
            total_steps += 1
    return total_bits / max(total_steps, 1)

In [ ]:

def run_pipeline(cfg, process, mode="forward"):
    """One process x one arm of the discrete transformer (models.DiscreteCausalDecoder),
    on the notebook's config.

    mode  "forward"   tril mask, predicts the next token;     compared with C+, occ_fw, T_theory_fw
          "backward"  triu mask, predicts the previous token; compared with C-, occ_bw, T_theory_bw
    """
    if mode not in ARM:
        raise ValueError(f"mode must be 'forward' or 'backward', got {mode!r}")
    spec = process_spec(cfg, process)
    # One cfg per run: V = spec["num_token"], K = num_states_mult * V, S = state_dim_mult * V.
    cfg = resolve_cfg(cfg, spec["num_token"])
    th = theory_for_arm(spec, mode)
    H = spec["entropy_rate"]

    print(f"==== 1) Generate observable data ({spec['tag']} | {mode} arm) ====")
    ds = SequenceDataset(generate(spec["kind"], spec["params"], cfg.num_samples, cfg.seq_len,
                                  spec["burn_in"], np.random.default_rng(cfg.data_seed)))
    # One seeded split, so both arms on this cfg see the same hold-out set.
    train_loader, test_loader = split_loader(ds, cfg.batch_size, cfg.test_ratio, seed=cfg.random_seed)
    ana_loader = tud.DataLoader(ds, batch_size=cfg.ana_batch, shuffle=False)
    test_indices = np.asarray(test_loader.dataset.indices)
    print(f"Vocabulary size V: {cfg.symbol_vocab_size} | state budget K: {cfg.num_states} | "
          f"state_dim S: {cfg.state_dim} | usage_beta: {cfg.usage_beta:.2e} | tau: {cfg.tau!r}")
    print(f"Sequences: {len(ds):,} x {ds.seq_len} scored tokens | "
          f"train {len(train_loader.dataset):,} / test {len(test_indices):,}")
    print(f"Closed forms: H_inf = {H:.4f} bits | C+ = {spec['C_plus']:.4f} | C- = {spec['C_minus']:.4f} | "
          f"true k = {spec['true_k_fw']} fw / {spec['true_k_bw']} bw | this arm: C = {th['C']:.4f}, k = {th['true_k']}")

    print(f"\n==== 2) Train the discrete transformer ({mode}) ====")
    set_seed(cfg.random_seed)
    torch.manual_seed(cfg.random_seed * 1000)
    t_start = time.time()
    with quiet(cfg.quiet_training):
        rec = train_model(
            train_loader, "discrete", val_loader=test_loader,
            num_token=cfg.symbol_vocab_size, d_model=cfg.d_model, max_len=ds.seq_len,
            max_epochs=cfg.max_epochs, lr=cfg.learning_rate, mode=mode, n_layers=cfg.n_layers,
            weight_decay=cfg.weight_decay, accelerator=cfg.accelerator,
            val_every_n_steps=cfg.val_every_n_steps,
            n_states=cfg.num_states, state_dim=cfg.state_dim, tau=cfg.tau, usage_beta=cfg.usage_beta,
        )
    model = to_cpu_for_analysis(rec.model)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Trained {type(model).__name__} ({n_params:,} parameters) in {time.time() - t_start:.0f} s, "
          f"{len(rec.step_loss)} gradient steps")
    history = {
        "step_at": [int(v) for v in rec.step_at], "step_loss": [float(v) for v in rec.step_loss],
        "val_at": [int(v) for v in rec.step_val_at], "val_loss": [float(v) for v in rec.step_val_loss],
        "epoch_loss": [float(v) for v in rec.epoch_loss],
    }

    print("\n==== 3) Evaluate against the entropy rate ====")
    test_ce, test_ppl = eval_ce(model, test_loader)
    divergence = diagnose_divergence(rec.step_loss)
    within_tol = bool(abs(test_ce - H) <= cfg.conv_tol)
    print(f"Test CE:      {test_ce:.6f} bits/token (perplexity {test_ppl:.4f})")
    print(f"H_inf:        {H:.6f} bits/token (closed form)")
    print(f"CE - H_inf:   {test_ce - H:+.6f}  -> {'converged' if within_tol else 'NOT converged'} "
          f"(conv_tol {cfg.conv_tol}){'  [DIVERGED after its minimum]' if divergence['diverged'] else ''}")

    # The true machine scored on the held-out sequences, in this arm's direction.
    tokens_te = arm_inputs(ds.seqs[test_indices], mode)
    T = tokens_te.shape[1]
    t0 = cfg.state_min_pos if mode == "forward" else T - 1 - cfg.state_min_pos
    true_start = th["true_machine"]["next_state"][0, tokens_te[:, t0]]
    true_ce = symbolic_cross_entropy(th["true_machine"], tokens_te, true_start, mode, t0)
    print(f"True-machine CE on the test sequences: {true_ce:.6f} bits/token")

    print("\n==== 4) Read the causal states off the bottleneck ====")
    report = causal_state_report(model, ana_loader, min_pos=cfg.state_min_pos)
    report["S_emp"] = float(report["S_emp"]) + 0.0        # a single state gives -0.0
    print(f"States used: {report['n_states_used']} / {report['n_states']} (true {th['true_k']})")
    print(f"S_emp = {report['S_emp']:.6f} bits vs C = {th['C']:.6f} bits | "
          f"H(state | token) = {report['h_state_given_token']:.6f} bits")
    print("Learned occupancy (visited, sorted):", np.round(np.sort(report["occupancy"][report["occupied"]])[::-1], 4))
    print("Theory occupancy (sorted):          ", np.round(np.sort(np.asarray(th["occupancy"]))[::-1], 4))

    print("\n==== 5) Learned state-to-state transition matrix vs the closed form ====")
    # Free-running generation: for a backward model the matrix is one step back in time,
    # the convention of T_theory_bw (processes.*_rev_transition_matrix).
    T_learned = transition_matrix_extraction(
        model, spec["kind"], spec["params"], burn_in=cfg.seq_len + 50,
        total_run=cfg.trans_total_run, window_size=cfg.seq_len,
        rng=np.random.default_rng(cfg.random_seed))
    comparison = compare_transition_matrix(
        T_learned, th["T_theory"], report["emissions"], th["true_machine"]["emission_probs"],
        weights=report["occupancy"])
    groups = {th["names"][t]: members for t, members in comparison["groups"].items()}
    print(f"{comparison['n_visited']} states visited in generation, {th['true_k']} in theory")
    print(f"Learned -> theoretical states (by emission row): {groups}")
    print(f"max |T_learned - T_theory| = {comparison['max_error']:.4f} | mean = {comparison['mean_error']:.4f}"
          + (f" | theoretical states with no learned state: {[th['names'][t] for t in comparison['missing']]}"
             if comparison["missing"] else ""))
    if comparison["permutation"] is not None:
        print(f"(match_permutation, the F3 comparison: max err {comparison['permutation'][2]:.4f})")

    print("\n==== 6) Empirical symbolic machine from the state trajectories ====")
    machine = empirical_symbolic_machine(model, ana_loader, cfg.state_min_pos)
    det = machine["determinism"]
    det_mean = float(np.nanmean(det)) if np.isfinite(det).any() else float("nan")
    weighted = machine["pair_counts"] / machine["pair_counts"].sum()
    det_weighted = float(np.nansum(np.where(np.isfinite(det), det * weighted, 0.0)))
    minimal_stationary = stationary_distribution_from_machine(machine)
    print(f"Machine size: {machine['next_state'].shape[0]} (true {th['true_k']}) | "
          f"determinism of (state, token) -> state: mean {det_mean:.4f}, visit-weighted {det_weighted:.4f}")

    print("\n==== 7) Score the symbolic machine on the test sequences ====")
    states_te, _ = hard_state_trajectories(model, test_loader)
    symbolic_start = machine["relabel"][states_te[:, t0]]
    symbolic_ce = symbolic_cross_entropy(machine, tokens_te, symbolic_start, mode, t0)
    print(f"Symbolic-machine test CE: {symbolic_ce:.6f} bits/token | neural {test_ce:.6f} | "
          f"true machine {true_ce:.6f} | H_inf {H:.6f}")

    summary_table = pd.DataFrame({
        "Metric": [
            "Test CE (bits)", "Entropy rate H_inf (bits)", "CE - H_inf",
            "Converged (|CE - H_inf| <= conv_tol)", "Diverged after minimum",
            "True-machine test CE (bits)", "Symbolic-machine test CE (bits)",
            "S_emp (bits)", "C (theory, bits)",
            "States used", "State budget K", "True machine size",
            "H(state | token) (bits)",
            "Transition max |err| (emission-matched)", "Transition mean |err|",
            "Theoretical states missing",
            "Symbolic machine determinism",
        ],
        "Value": [
            test_ce, H, test_ce - H,
            within_tol, bool(divergence["diverged"]),
            true_ce, symbolic_ce,
            report["S_emp"], th["C"],
            report["n_states_used"], report["n_states"], th["true_k"],
            report["h_state_given_token"],
            comparison["max_error"], comparison["mean_error"],
            len(comparison["missing"]),
            det_weighted,
        ],
    })

    return {
        "cfg": cfg, "process": process, "mode": mode, "spec": spec, "theory": th,
        "model": model, "history": history, "dataset": ds, "test_indices": test_indices,
        "train_loader": train_loader, "test_loader": test_loader, "ana_loader": ana_loader,
        "report": report, "raw_state_counts": report["counts"],
        "transition": T_learned, "transition_comparison": comparison,
        "machine": machine, "minimal_machine": machine, "minimal_stationary": minimal_stationary,
        "true_machine": th["true_machine"], "true_stationary": th["occupancy"],
        "summary_table": summary_table,
        "metrics": {
            "test_ce": test_ce, "test_ppl": test_ppl, "entropy_rate": H, "gap": test_ce - H,
            "within_tol": within_tol, "diverged": bool(divergence["diverged"]),
            "true_ce": true_ce, "symbolic_ce": symbolic_ce,
            "S_emp": report["S_emp"], "C": th["C"], "k": report["n_states_used"], "true_k": th["true_k"],
            "h_state_given_token": report["h_state_given_token"],
            "transition_max_err": comparison["max_error"], "transition_mean_err": comparison["mean_error"],
            "transition_missing": len(comparison["missing"]), "determinism": det_weighted,
        },
    }

def run_both_arms(cfg, process):
    """The forward and backward discrete models on the same realisation and hold-out split."""
    runs = {}
    for mode in ("forward", "backward"):
        print("\n" + "#" * 78 + f"\n# {process}: {mode}\n" + "#" * 78)
        runs[mode] = run_pipeline(cfg, process, mode=mode)
        cleanup()
    return runs

def arms_table(runs):
    """One row per arm: the extracted states and transition matrix against theory."""
    rows = []
    for mode, r in runs.items():
        m = r["metrics"]
        rows.append({
            "arm": mode, "test CE (bits)": m["test_ce"], "H_inf": m["entropy_rate"], "converged": m["within_tol"],
            "k found": m["k"], "true k": m["true_k"], "S_emp (bits)": m["S_emp"], "C (bits)": m["C"],
            "S_emp - C": m["S_emp"] - m["C"], "T max |err|": m["transition_max_err"],
            "T mean |err|": m["transition_mean_err"], "T states missing": m["transition_missing"],
        })
    return pd.DataFrame(rows)

In [ ]:

_BLUE, _ORNG, _GREY, _INK = "#3B6EA5", "#C4703A", "#D8DEE6", "#3C4653"     # figures.py palette

def plot_training_curves(results):
    history, H, tol = results["history"], results["spec"]["entropy_rate"], results["cfg"].conv_tol
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].axhspan(H - tol, H + tol, color=_GREY, alpha=0.55, lw=0, label="H_inf +- conv_tol")
    axes[0].axhline(H, color=_INK, lw=1.2, ls=(0, (5, 2)))
    axes[0].plot(history["step_at"], history["step_loss"], color=_BLUE, lw=0.8, alpha=0.6,
                 label="Train objective (per step)")
    if history["val_at"]:
        axes[0].plot(history["val_at"], history["val_loss"], color=_ORNG, lw=2.0, label="Validation CE")
    axes[0].set_title(f"{results['spec']['tag']} ({results['mode']}): cross-entropy vs H_inf = {H:.4f}")
    axes[0].set_xlabel("Gradient step")
    axes[0].set_ylabel("bits / token")
    axes[0].grid(alpha=0.25)
    axes[0].legend()

    if history["val_at"]:
        gap = np.maximum(np.asarray(history["val_loss"]) - H, 1e-3)
        axes[1].semilogy(history["val_at"], gap, color=_ORNG, lw=2.0, label="Validation CE - H_inf")
    axes[1].axhline(tol, color=_INK, lw=1.0, ls=":", label=f"conv_tol = {tol}")
    axes[1].set_title("Gap to the entropy rate (log scale)")
    axes[1].set_xlabel("Gradient step")
    axes[1].set_ylabel("bits / token")
    axes[1].grid(alpha=0.25, which="both")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

def plot_state_counts(counts, title="Hard latent-state counts on the analysis sequences"):
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(np.arange(len(counts)), counts)
    ax.set_title(title)
    ax.set_xlabel("Bottleneck state")
    ax.set_ylabel("Count")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

def plot_state_occupancy(results, title=None):
    """Learned occupancy of the visited states, sorted, laid against the theoretical
    occupancy (identified only up to a permutation of the state labels)."""
    report, th = results["report"], results["theory"]
    occ = np.asarray(report["occupancy"])
    learned = np.sort(occ[occ > 0])[::-1]
    theory = np.sort(np.asarray(th["occupancy"]))[::-1]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(np.arange(len(learned)), learned, width=0.62, color=_BLUE, label="learned")
    ax.plot(np.arange(len(theory)), theory, "o", ms=7, color=_ORNG, label="theory")
    ax.set_xticks(range(max(len(learned), len(theory))))
    ax.set_xlabel("State (sorted by occupancy)")
    ax.set_ylabel("Occupancy")
    ax.set_title(title or f"{results['mode']}: S_emp = {report['S_emp']:.3f} / C = {th['C']:.3f} bits   "
                          f"k = {report['n_states_used']}/{report['n_states']} (true {th['true_k']})")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()

def plot_emission_tables(results, title="Emission tables"):
    """P(token | state) of the visited bottleneck states beside the true machine's."""
    machine, true = results["machine"], results["true_machine"]
    learned = np.asarray(machine["emission_probs"])
    truth = np.asarray(true["emission_probs"])
    token_label = "next token" if results["mode"] == "forward" else "previous token"

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, mat, name, ids in ((axes[0], learned, "learned", machine["state_ids"]),
                               (axes[1], truth, "true machine", results["theory"]["names"])):
        im = ax.imshow(mat, cmap="Blues", vmin=0, vmax=1, aspect="auto")
        ax.set_yticks(range(mat.shape[0]))
        ax.set_yticklabels([str(k) for k in ids], fontsize=8)
        ax.set_xticks(range(mat.shape[1]))
        ax.set_xlabel(token_label)
        ax.set_ylabel("state" if name == "learned" else "true state")
        ax.set_title(f"{name}: {mat.shape[0]} states")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

def plot_emission_ladder_comparison(machine, true_machine_, title="Emission profile comparison"):
    learned = np.asarray(machine["emission_probs"], dtype=np.float64)
    true = np.asarray(true_machine_["emission_probs"], dtype=np.float64)
    alphabet_size = true.shape[1]

    fig, axes = plt.subplots(1, alphabet_size, figsize=(5 * alphabet_size, 4), squeeze=False)
    axes = axes.ravel()

    for x in range(alphabet_size):
        learned_sorted = np.sort(learned[:, x])
        true_sorted = np.sort(true[:, x])
        axes[x].plot(np.arange(len(true_sorted)), true_sorted, marker="o", label=f"True sorted P({x})")
        axes[x].plot(np.arange(len(learned_sorted)), learned_sorted, marker="s", label=f"Learned sorted P({x})")
        axes[x].set_title(f"Symbol {x}")
        axes[x].set_xlabel(f"State index after sorting by P({x})")
        axes[x].set_ylabel(f"P(symbol = {x})")
        axes[x].grid(alpha=0.25)
        axes[x].legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

def _heat(ax, M, ticks, title):
    im = ax.imshow(M, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    K = M.shape[0]
    fs = 8 if K <= 8 else 5
    for i in range(K):
        for j in range(K):
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center",
                    fontsize=fs, color="white" if M[i, j] > 0.55 else _INK)
    ax.set_xticks(range(K)); ax.set_yticks(range(K))
    ax.set_xticklabels([str(t) for t in ticks], fontsize=7)
    ax.set_yticklabels([str(t) for t in ticks], fontsize=7)
    ax.set_xlabel("to"); ax.set_ylabel("from")
    ax.set_title(title, fontsize=10)
    return im

def _draw_transition_comparison(axes, results):
    """Three panels on one row: the learned matrix aggregated onto the theoretical states,
    the closed form, and their absolute difference (extraction.compare_transition_matrix)."""
    cmp, th, mode = results["transition_comparison"], results["theory"], results["mode"]
    names = th["names"]
    ticks = [f"{names[t]}\n<- {cmp['groups'][t] if cmp['groups'][t] else 'none'}" for t in range(len(names))]
    _heat(axes[0], cmp["aggregated"], ticks,
          f"{mode} (learned, {cmp['n_visited']} states -> {len(names)})")
    _heat(axes[1], cmp["theory"], names, f"{mode} (theory)")
    err = np.nan_to_num(cmp["abs_error"], nan=0.0)
    im = axes[2].imshow(err, cmap="Reds", vmin=0, vmax=max(0.05, float(err.max())), aspect="auto")
    K = err.shape[0]
    for i in range(K):
        for j in range(K):
            if np.isfinite(cmp["abs_error"][i, j]):
                axes[2].text(j, i, f"{err[i, j]:.2f}", ha="center", va="center", fontsize=8 if K <= 8 else 5,
                             color="white" if err[i, j] > 0.6 * max(0.05, float(err.max())) else _INK)
    axes[2].set_xticks(range(K)); axes[2].set_yticks(range(K))
    axes[2].set_xticklabels(names, fontsize=7); axes[2].set_yticklabels(names, fontsize=7)
    axes[2].set_xlabel("to"); axes[2].set_ylabel("from")
    missing = f"   missing: {[names[t] for t in cmp['missing']]}" if cmp["missing"] else ""
    axes[2].set_title(f"|learned - theory|   max {cmp['max_error']:.3f}, mean {cmp['mean_error']:.3f}{missing}",
                      fontsize=10)
    return im

def plot_transition_comparison(results, title=None):
    """Learned T[i][j] = P(s' = j | s = i) from free-running generation against the closed
    form of this arm.  Learned states are mapped onto theoretical ones by their emission
    rows, so a causal state the bottleneck split into several is re-merged for the comparison."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
    _draw_transition_comparison(axes, results)
    fig.suptitle(title or f"{results['spec']['tag']} ({results['mode']}): state-to-state transition matrix, "
                          f"learned vs theory")
    plt.tight_layout()
    plt.show()

def plot_transition_comparison_arms(runs, title=None):
    """The same comparison for the forward and backward arms, one row each: the forward
    matrix against T_theory_fw and the backward one against T_theory_bw."""
    modes = [m for m in ("forward", "backward") if m in runs]
    fig, axes = plt.subplots(len(modes), 3, figsize=(16, 4.6 * len(modes)), squeeze=False)
    for row, mode in enumerate(modes):
        _draw_transition_comparison(axes[row], runs[mode])
    tag = runs[modes[0]]["spec"]["tag"]
    fig.suptitle(title or f"{tag}: learned vs theoretical state-to-state transition matrices, both arms")
    plt.tight_layout()
    plt.show()

def plot_transition_heatmaps(machine, title_prefix="Transition structure"):
    next_state = machine["next_state"]
    alphabet_size = next_state.shape[1]
    fig, axes = plt.subplots(1, alphabet_size, figsize=(5 * alphabet_size, 4), squeeze=False)
    axes = axes.ravel()

    for symbol in range(alphabet_size):
        mat = np.zeros((next_state.shape[0], next_state.shape[0]), dtype=np.float64)
        for s in range(next_state.shape[0]):
            mat[s, int(next_state[s, symbol])] = 1.0
        im = axes[symbol].imshow(mat, aspect="auto")
        axes[symbol].set_title(f"{title_prefix}: symbol {symbol}")
        axes[symbol].set_xlabel("Next state")
        axes[symbol].set_ylabel("Current state")
        fig.colorbar(im, ax=axes[symbol], fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

def plot_determinism(machine, title="Determinism of (state, token) -> next state"):
    """Fraction of visits agreeing with the majority successor; 1 everywhere for a unifilar machine."""
    det = np.asarray(machine["determinism"], dtype=np.float64)
    fig, ax = plt.subplots(figsize=(max(6, 0.6 * det.shape[1] + 3), 0.45 * det.shape[0] + 2))
    im = ax.imshow(np.nan_to_num(det, nan=0.0), cmap="Blues", vmin=0, vmax=1, aspect="auto")
    for i in range(det.shape[0]):
        for j in range(det.shape[1]):
            if np.isfinite(det[i, j]):
                ax.text(j, i, f"{det[i, j]:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if det[i, j] > 0.55 else _INK)
    ax.set_xticks(range(det.shape[1])); ax.set_yticks(range(det.shape[0]))
    ax.set_xlabel("token"); ax.set_ylabel("state")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

def plot_symbolic_machine_graph(machine, stationary_dist=None, title="Extracted symbolic machine"):
    next_state = machine["next_state"]
    emission_probs = machine["emission_probs"]
    num_states, alphabet_size = next_state.shape

    G = nx.DiGraph()
    edge_labels = {}
    self_loops = {s: [] for s in range(num_states)}
    for s in range(num_states):
        for x in range(alphabet_size):
            p = float(emission_probs[s, x])
            if p <= 0.01:
                continue
            ns = int(next_state[s, x])
            lbl = f"{x}|{p:.2f}"
            if ns == s:                       # a self-loop is drawn under the node, so it is labelled on the node
                self_loops[s].append(lbl)
            key = (s, ns)
            if key in edge_labels:
                edge_labels[key] += f"\n{lbl}"
            else:
                G.add_edge(s, ns)
                edge_labels[key] = lbl

    for s in range(num_states):
        label = f"s{s}"
        if stationary_dist is not None:
            label += f"\nπ={stationary_dist[s]:.3f}"
        if self_loops[s]:
            label += "\n↺ " + ", ".join(self_loops[s])
        G.add_node(s, label=label)

    scale = max(1.0, num_states / 10.0)
    fig_w = max(12, int(8 * scale))
    fig_h = max(8, int(6 * scale))

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    try:
        pos = nx.kamada_kawai_layout(G)
    except Exception:
        pos = nx.spring_layout(G, seed=7, k=2.5 / max(1, num_states ** 0.5), iterations=200)

    node_size = max(800, 2400 - 80 * num_states)
    font_size = max(6, 10 - num_states // 6)

    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_size,
                           node_color="lightsteelblue", edgecolors="steelblue", linewidths=1.5)
    nx.draw_networkx_labels(G, pos, ax=ax, labels=nx.get_node_attributes(G, "label"),
                            font_size=font_size, font_weight="bold")
    nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>",
                           arrowsize=14, width=1.2, edge_color="gray",
                           connectionstyle="arc3,rad=0.1", min_source_margin=15, min_target_margin=15)

    edge_font = max(5, font_size - 1)
    nx.draw_networkx_edge_labels(G, pos, ax=ax,
                                 edge_labels={k: v for k, v in edge_labels.items() if k[0] != k[1]},
                                 font_size=edge_font, label_pos=0.35,
                                 bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8))

    ax.set_title(title, fontsize=13)
    ax.axis("off")
    fig.tight_layout()
    plt.show()

def plot_complexity(runs, title=None):
    """Theory vs the discrete extractor per arm: C against S_emp, with the state counts."""
    modes = [m for m in ("forward", "backward") if m in runs]
    fig, axes = plt.subplots(1, len(modes), figsize=(5 * len(modes), 4.2), squeeze=False)
    for ax, mode in zip(axes.ravel(), modes):
        r = runs[mode]
        m, th = r["metrics"], r["theory"]
        ax.bar([0, 1], [th["C"], m["S_emp"]], width=0.6, color=[_GREY, _BLUE])
        ax.set_xticks([0, 1])
        ax.set_xticklabels([f"theory\nk = {th['true_k']}",
                            f"discrete\nk = {m['k']}" + ("" if m["within_tol"] else "\n[not converged]")], fontsize=9)
        ax.set_ylabel("bits")
        ax.set_title(f"{mode}   C = {th['C']:.3f} / S_emp = {m['S_emp']:.3f}", fontsize=10)
        ax.grid(axis="y", alpha=0.25)
    tag = runs[modes[0]]["spec"]["tag"]
    fig.suptitle(title or f"{tag}: statistical complexity, theory vs extracted")
    plt.tight_layout()
    plt.show()

def plot_arm(results):
    """Every plot for one run, in the order of the pipeline steps."""
    tag, mode = results["spec"]["tag"], results["mode"]
    plot_training_curves(results)
    plot_state_counts(results["raw_state_counts"],
                      title=f"{tag} ({mode}): bottleneck-state counts on the analysis sequences")
    plot_state_occupancy(results)
    plot_emission_tables(results, title=f"{tag} ({mode}): emission tables")
    plot_emission_ladder_comparison(
        results["machine"], results["true_machine"],
        title=f"Empirical symbolic machine vs. true {tag} ({mode}) emission profiles")
    plot_transition_comparison(results)
    plot_transition_heatmaps(results["machine"], title_prefix=f"{tag} ({mode}) empirical machine")
    plot_determinism(results["machine"], title=f"{tag} ({mode}): determinism of (state, token) -> next state")
    plot_symbolic_machine_graph(
        results["machine"], stationary_dist=results["minimal_stationary"],
        title=f"Empirical symbolic machine of the {mode} model"
              + (" (edges read one step back in time)" if mode == "backward" else ""))
    display(results["summary_table"])

## Running it

`run_pipeline` trains one model (about 30 s on MPS at the default 500 x 300 tokens, 150 epochs) and prints the theory it is compared against at every step. Training on these processes is **bimodal**: a run either converges to `H_inf` (the `Converged` row of the summary) or collapses -- cross-entropy far above `H_inf`, a single state used, `Diverged after minimum = True`. That is a property of the training (both processes contain deterministic transitions, so the cross-entropy has no finite optimum; see `training.diagnose_divergence`), not of the extraction. MPS is not bit-reproducible, so re-running the same cell can land in the other mode; `accelerator = "cpu"` is exactly repeatable at roughly 6x the wall clock. When a run collapses, re-run it or change `random_seed`, and read the state plots only from converged runs.

## Coin process: forward and backward arms

In [ ]:
coin_fw = run_pipeline(cfg, "coin", mode="forward")

In [ ]:
plot_arm(coin_fw)

In [ ]:
coin_bw = run_pipeline(cfg, "coin", mode="backward")

In [ ]:
plot_arm(coin_bw)

In [ ]:
# Both arms against their closed forms: the forward matrix against T_theory_fw, the
# backward one against T_theory_bw (coin_rev_transition_matrix), plus C+/C- against S_emp.
coin_runs = {"forward": coin_fw, "backward": coin_bw}
plot_transition_comparison_arms(coin_runs)
plot_complexity(coin_runs)
display(arms_table(coin_runs))

## Flower process: forward and backward arms

V = n + m tokens and K = num_states_mult · V states. Forward the theory has n + 1 states (R and one per die); backward it has 1 + the number of *distinguishable* outcomes (S and one class per distinct posterior over dice), which for generic Dirichlet dice is m + 1.

With V = 11 the flower needs more gradient steps than the coin at the same learning rate: at 150 epochs the forward arm stops at 3 of the 4 states with a gap of about 0.15 bits, at 300 it converges (4 states, `S_emp` on C+, transition error about 0.01), so the flower cells run at `max_epochs = 300`. The coin is left at 150 — its backward arm converges and then diverges past `H_inf` if trained longer. The backward flower arm (9 states) did not converge at 150, 300 or 400 epochs in the runs used to write this notebook; it finds 3–4 of the 9 states, and the comparison below reports which theoretical states are missing rather than forcing a match.

In [ ]:
cfg_flower = replace(cfg, max_epochs=300)      # V = 11: twice the steps of the coin at the same lr

In [ ]:
flower_fw = run_pipeline(cfg_flower, "flower", mode="forward")

In [ ]:
plot_arm(flower_fw)

In [ ]:
flower_bw = run_pipeline(cfg_flower, "flower", mode="backward")

In [ ]:
plot_arm(flower_bw)

In [ ]:
flower_runs = {"forward": flower_fw, "backward": flower_bw}
plot_transition_comparison_arms(flower_runs)
plot_complexity(flower_runs)
display(arms_table(flower_runs))

In [ ]:
# Everything on one table: both processes, both arms.
summary = pd.concat([arms_table(coin_runs).assign(process="coin"),
                     arms_table(flower_runs).assign(process="flower")], ignore_index=True)
display(summary.set_index(["process", "arm"]))